In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Customer_Churn_Prediction\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Customer_Churn_Prediction'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelPromotionConfig:
    root_dir: Path
    metrics_file_path: Path
    model_file_path: Path
    production_model_path: Path
    registered_model_name: str
    target_stage: str
    mlflow_uri: str
    promote_metric: str
    promote_threshold: float
    archive_existing_versions: bool
    copy_local_model: bool     

In [6]:
from src.customer_churn_prediction.constants import *
from src.customer_churn_prediction.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_promotion_config(self) -> ModelPromotionConfig:
        config = self.config.model_promotion

        create_directories([config.root_dir])

        model_promotion_config = ModelPromotionConfig(
            root_dir=config.root_dir,
            metrics_file_path=config.metrics_file_path,
            model_file_path=config.model_file_path,
            production_model_path=config.production_model_path,
            registered_model_name=config.registered_model_name,
            target_stage=config.target_stage,
            mlflow_uri=config.mlflow_uri,
            promote_metric=config.promote_metric,
            promote_threshold=float(config.promote_threshold),
            archive_existing_versions=bool(config.archive_existing_versions),
            copy_local_model=bool(config.copy_local_model)
        )

        return model_promotion_config    

In [8]:
import shutil
from pathlib import Path
import mlflow
from mlflow.tracking import MlflowClient
from src.customer_churn_prediction import logger
from src.customer_churn_prediction.utils.common import load_json, save_json

In [9]:
class ModelPromotion:
    def __init__(self, config: ModelPromotionConfig):
        self.config = config
        create_directories([self.config.root_dir])

    def load_metrics(self) -> dict:
        metrics_path = Path(self.config.metrics_file_path)
        if not metrics_path.exists():
            raise FileNotFoundError(f"Evaluation metrics not found: {metrics_path}")

        logger.info(f"Loading evaluation metrics from: {metrics_path}")
        metrics = load_json(path=metrics_path)
        return dict(metrics)

    def should_promote(self, metrics: dict) -> bool:
        promote_metric = self.config.promote_metric
        if promote_metric not in metrics:
            raise KeyError(f"Promotion metric '{promote_metric}' not found in evaluation metrics")

        score = float(metrics[promote_metric])
        logger.info(f"Promotion metric '{promote_metric}': {score}")
        promoted = score >= self.config.promote_threshold
        logger.info(
            f"Promotion threshold: {self.config.promote_threshold} -> {'PASS' if promoted else 'FAIL'}"
        )
        return promoted

    def get_latest_registered_version(self) -> object:
        client = MlflowClient(registry_uri=self.config.mlflow_uri)
        logger.info(f"Searching registered model versions for: {self.config.registered_model_name}")
        versions = client.search_model_versions(f"name='{self.config.registered_model_name}'")
        if not versions:
            raise ValueError(f"No registered versions found for model: {self.config.registered_model_name}")

        versions_sorted = sorted(versions, key=lambda v: int(v.version))
        latest = versions_sorted[-1]
        logger.info(
            f"Latest registered model version: {latest.version}, stage: {latest.current_stage}, run_id: {latest.run_id}"
        )
        return latest

    def transition_to_stage(self, version: str) -> str:
        client = MlflowClient(registry_uri=self.config.mlflow_uri)
        logger.info(
            f"Transitioning model '{self.config.registered_model_name}' version {version} "
            f"to stage '{self.config.target_stage}'"
        )
        client.transition_model_version_stage(
            name=self.config.registered_model_name,
            version=version,
            stage=self.config.target_stage,
            archive_existing_versions=self.config.archive_existing_versions
        )
        logger.info(f"Model version {version} transitioned to {self.config.target_stage}")
        return version

    def save_production_copy(self):
        if not self.config.copy_local_model:
            logger.info("Copying local model to production path is disabled.")
            return None

        source_path = Path(self.config.model_file_path)
        production_path = Path(self.config.production_model_path)
        if not source_path.exists():
            raise FileNotFoundError(f"Trained model file not found: {source_path}")

        production_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, production_path)
        logger.info(f"Copied model to production path: {production_path}")
        return str(production_path)

    def promote(self) -> dict:
        metrics = self.load_metrics()
        if not self.should_promote(metrics):
            msg = (
                f"Promotion criteria not met. "
                f"Metric {self.config.promote_metric} must be >= {self.config.promote_threshold}"
            )
            logger.info(msg)
            return {
                "promoted": False,
                "reason": msg,
                "metric": self.config.promote_metric,
                "value": float(metrics.get(self.config.promote_metric, 0.0))
            }

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_registry_uri(self.config.mlflow_uri)
        latest_version = self.get_latest_registered_version()
        promoted_version = self.transition_to_stage(latest_version.version)
        production_path = self.save_production_copy()

        promotion_summary = {
            "promoted": True,
            "registered_model_name": self.config.registered_model_name,
            "version": promoted_version,
            "stage": self.config.target_stage,
            "metric": self.config.promote_metric,
            "metric_value": float(metrics[self.config.promote_metric]),
            "production_model_path": production_path,
        }

        summary_file = Path(self.config.root_dir) / "promotion_summary.json"
        save_json(path=summary_file, data=promotion_summary)
        logger.info(f"Promotion summary saved to: {summary_file}")

        return promotion_summary

In [ ]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/ajaychaudhary8104/End_to_End_ML_project_for_Customer_Churn_Prediction.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="ajaychaudhary8104"
os.environ["MLFLOW_TRACKING_PASSWORD"]="password"

In [12]:
try:    
    config = ConfigurationManager()
    model_promotion_config = config.get_model_promotion_config()
    model_promotion = ModelPromotion(config=model_promotion_config)

    logger.info("Evaluating promotion criteria for production deployment...")
    promotion_result = model_promotion.promote()

    logger.info(f"Promotion result: {promotion_result}")
except Exception as e:
    logger.error(f"Error during model promotion: {e}")
    raise e    

[2026-05-21 00:02:16,590: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-21 00:02:16,595: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-21 00:02:16,597: INFO: common: created directory at: artifacts]
[2026-05-21 00:02:16,599: INFO: common: created directory at: artifacts/model_promotion]
[2026-05-21 00:02:16,600: INFO: common: created directory at: artifacts/model_promotion]
[2026-05-21 00:02:16,601: INFO: 4119663335: Evaluating promotion criteria for production deployment...]
[2026-05-21 00:02:16,602: INFO: 492532602: Loading evaluation metrics from: artifacts\model_evaluation\model_evaluation_metrics.json]
[2026-05-21 00:02:16,604: INFO: common: json file loaded succesfully from: artifacts\model_evaluation\model_evaluation_metrics.json]
[2026-05-21 00:02:16,605: INFO: 492532602: Promotion metric 'f1': 0.7878643394838212]
[2026-05-21 00:02:16,606: INFO: 492532602: Promotion threshold: 0.77 -> PASS]
[2026-05-21 00:02:16,608: INFO: 4925326

C:\Users\Hp\AppData\Local\Temp\ipykernel_8388\492532602.py:48: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.1/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


[2026-05-21 00:02:22,004: INFO: 492532602: Model version 3 transitioned to Production]
[2026-05-21 00:02:22,004: INFO: 492532602: Copied model to production path: artifacts\model_promotion\production_model.pkl]
[2026-05-21 00:02:22,004: INFO: common: json file saved at: artifacts\model_promotion\promotion_summary.json]
[2026-05-21 00:02:22,004: INFO: 492532602: Promotion summary saved to: artifacts\model_promotion\promotion_summary.json]
[2026-05-21 00:02:22,020: INFO: 4119663335: Promotion result: {'promoted': True, 'registered_model_name': 'XGBClassifier', 'version': '3', 'stage': 'Production', 'metric': 'f1', 'metric_value': 0.7878643394838212, 'production_model_path': 'artifacts\\model_promotion\\production_model.pkl'}]
